# NB4 — Core-7 V2 + Negative V1 → scorer-ready dataset

NB4 chỉ tạo negative sau khi NB3 PASS **và** SHA-256 của exact input hiện tại khớp report NB3.

Core-7 mapping version: `core7-v2`. Negative protocol vẫn là `negative-v1 = same_category_different_kit`. Trước official freeze, Git commit phải tồn tại và working tree phải sạch.

## 1. Runtime portable

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
AUTO_CLONE_REPO = False

def find_repo_root(start: Path = Path.cwd()):
    explicit = os.environ.get("FASHION_PROJECT_ROOT")
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        if (candidate / "src/data/prepare_core7_dataset.py").exists():
            return candidate
        raise FileNotFoundError(f"FASHION_PROJECT_ROOT không hợp lệ: {candidate}")
    current = start.expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src/data/prepare_core7_dataset.py").exists():
            return candidate
    return None

REPO_ROOT = find_repo_root()
if REPO_ROOT is None and AUTO_CLONE_REPO:
    REPO_ROOT = (Path.cwd() / "opisoverated").resolve()
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
if REPO_ROOT is None:
    raise RuntimeError("Không tìm thấy repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.build_core7_scorer_dataset import inspect_git_provenance
from src.data.runtime_paths import load_runtime_paths
RUNTIME_PATHS = load_runtime_paths(repo_root=REPO_ROOT)

GIT_PROVENANCE = inspect_git_provenance(REPO_ROOT)
GIT_COMMIT = GIT_PROVENANCE["git_commit"]
GIT_TREE_CLEAN = GIT_PROVENANCE["git_tree_clean"]

print("Repo root     :", REPO_ROOT)
print("Artifact root :", RUNTIME_PATHS.artifact_root)
print("Git commit    :", GIT_COMMIT or "unavailable")
print("Git tree clean:", GIT_TREE_CLEAN)
if GIT_PROVENANCE["dirty_entry_examples"]:
    print("Dirty entries   :", GIT_PROVENANCE["dirty_entry_examples"])
if GIT_PROVENANCE["error"]:
    print("Git error       :", GIT_PROVENANCE["error"])
if not GIT_COMMIT or not GIT_TREE_CLEAN:
    raise RuntimeError(
        "BLOCKED: official freeze cần Git commit hợp lệ và working tree sạch. "
        "Commit/push code trước rồi chạy lại NB4."
    )


## 2. Exact NB3 inputs

NB4 cần cả cache và manifest hiện tại để tự hash lại. Chỉ đọc `pass=true` là không đủ.

In [ ]:
CORE7_DIR = RUNTIME_PATHS.core7_dir
MAPPING_PATH = REPO_ROOT / "configs/category_mapping_core7_v2.json"
OUTPUT_DIR = RUNTIME_PATHS.scorer_ready_dir
CACHE_PATH = RUNTIME_PATHS.embedding_cache
MANIFEST_PATH = RUNTIME_PATHS.embedding_manifest
EMBEDDING_REPORT = CORE7_DIR / "core7_embedding_validation_report.json"
SEED = 42
ALLOW_OVERWRITE = False

required = [MAPPING_PATH, EMBEDDING_REPORT, CACHE_PATH, MANIFEST_PATH]
for split in ("train", "valid", "test"):
    required.extend([
        CORE7_DIR / f"category_clean_{split}.jsonl",
        CORE7_DIR / f"core7_item_metadata_v1_{split}.jsonl",
    ])
for path in required:
    print(path, "exists=", path.is_file())
    if not path.is_file():
        raise FileNotFoundError(path)


## 3. Build scorer-ready V2

`build_scorer_dataset_v2` verify SHA-256 trước khi sinh negative. Nếu NB2/NB3 input đã thay đổi, cell này hard-fail và yêu cầu chạy lại NB3.

In [ ]:
from src.data.build_core7_scorer_dataset import build_scorer_dataset_v2

result = build_scorer_dataset_v2(
    data_dir=CORE7_DIR,
    output_dir=OUTPUT_DIR,
    embedding_report_path=EMBEDDING_REPORT,
    category_mapping_path=MAPPING_PATH,
    embedding_cache_path=CACHE_PATH,
    embedding_manifest_path=MANIFEST_PATH,
    repo_root=REPO_ROOT,
    seed=SEED,
    overwrite=ALLOW_OVERWRITE,
)
print("STATUS         :", result["status"])
print("DATASET VERSION:", result["dataset_version"])
print("MAPPING VERSION:", result["category_mapping_version"])
print("NEGATIVE VERSION:", result["negative_version"])


## 4. Validation summary

In [ ]:
final_report = result["final_validation"]
print("Embedding input hashes :", final_report["embedding_input_verification"]["pass"])
print("Embedding gate         :", final_report["embedding_validation_pass"])
print("Negative sampling gate :", final_report["negative_sampling_pass"])
print("Source-kit cross-split :", final_report["source_kit_cross_split_count"])
print("Item cross-split       :", final_report["item_cross_split_count"])
print("Duplicate sample IDs   :", final_report["global_duplicate_sample_id_count"])
print("FINAL STATUS           :", result["status"])


## 5. Expected outputs

```text
scorer_ready_v2/
├── negative_v1_{train,valid,test}.jsonl
├── scorer_ready_v2_{train,valid,test}.jsonl
├── negative_sampling_v1_*_report.json
├── final_validation_v2.json
├── split_manifest_v2.json
└── dataset_manifest_v2.json
```

Negative vẫn là protocol V1; dataset/mapping được bump sang V2 vì category semantics đã đổi.